# Agent Sample

In [1]:
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
from anthropic import Anthropic
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
client = Anthropic()
model = "claude-sonnet-4-0"

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].text

In [14]:
# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

# Get Claude's response
answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)



In [15]:
# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_answer = chat(messages)

In [16]:
final_answer

'Quantum computers use quantum bits (qubits) that can exist in multiple states simultaneously, unlike classical bits that are either 0 or 1, enabling them to perform many calculations in parallel.'

In [ ]:
messages = []

while True:
    user_input = input("> ")
    print(f"You: {user_input}")
    add_user_message(messages, user_input)
    
    response = chat(messages)
    print("---")
    print(f"Claude: {response}")
    print("---")
    add_assistant_message(messages, response)

## System Prompts

In [18]:
client = Anthropic()
model = "claude-sonnet-4-0"

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

system_prompt = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

def chat(**kwargs):
    messages = kwargs.get("messages", [])
    system = kwargs.get("system", False)
    temperature = kwargs.get("temperature", None)
    stop_sequences = kwargs.get("stop_sequences", None)

    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    
    if system:
        params["system"] = system_prompt
    
    if temperature:
        params["temperature"] = temperature

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    print(f"parametros: {params}")

    message = client.messages.create(**params)
    
    return message.content[0].text


In [ ]:
messages = []

while True:
    user_input = input("> ")
    print(f"You: {user_input}")
    add_user_message(messages, user_input)
    
    response = chat(messages, True)
    print("---")
    print(f"Claude: {response}")
    print("---")
    add_assistant_message(messages, response)

## Response Stream

In [20]:
client = Anthropic()
model = "claude-sonnet-4-0"

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

system_prompt = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

def chat_stream(messages, system: bool = False, temperature: float = 1.0):

    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature        
    }
    
    if system:
        params["system"] = system_prompt
    
    print(f"parametros: {params}")

    with client.messages.stream(**params) as stream:
        for text in stream.text_stream:
            print(text, end="")
    
    # get the full message content after streaming is done
    final_message = stream.get_final_message()
    return final_message
    

In [13]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

response = chat_stream(messages, system=True, temperature=0.7)
add_assistant_message(messages, response)


parametros: {'model': 'claude-sonnet-4-0', 'max_tokens': 1000, 'messages': [{'role': 'user', 'content': 'Write a 1 sentence description of a fake database'}], 'temperature': 0.7, 'system': "\nYou are a patient math tutor.\nDo not directly answer a student's questions.\nGuide them to a solution step by step.\n"}
Here's a one-sentence description of a fake database:

**PetPalace Database** - A comprehensive pet store inventory system that tracks animal information (species, age, health records), customer profiles, adoption histories, and supply management across multiple store locations.

Now, what specific aspect of this database would you like to explore mathematically? We could work on calculating adoption rates, inventory turnover, or perhaps some probability problems related to pet characteristics!

In [14]:
messages

[{'role': 'user',
  'content': 'Write a 1 sentence description of a fake database'},
 {'role': 'assistant',
  'content': ParsedMessage(id='msg_01BrJ9uGCyAUfPYdxKATLL3H', container=None, content=[ParsedTextBlock(citations=None, text="Here's a one-sentence description of a fake database:\n\n**PetPalace Database** - A comprehensive pet store inventory system that tracks animal information (species, age, health records), customer profiles, adoption histories, and supply management across multiple store locations.\n\nNow, what specific aspect of this database would you like to explore mathematically? We could work on calculating adoption rates, inventory turnover, or perhaps some probability problems related to pet characteristics!", type='text', parsed_output=None)], model='claude-sonnet-4-20250514', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation

## Dealing with Structured Data

In [27]:
messages = []
prompt ="""
Generate three different sample aws cli commands. Each should be very short.
"""

add_user_message(messages, prompt)
add_assistant_message(messages, "Here are all three commands in a single block without any comments:\n```bash")

text = chat(
    messages=messages, 
    system=False, 
    stop_sequences=["```"]
    )
text.strip()

parametros: {'model': 'claude-sonnet-4-0', 'max_tokens': 1000, 'messages': [{'role': 'user', 'content': '\nGenerate three different sample aws cli commands. Each should be very short.\n'}, {'role': 'assistant', 'content': 'Here are all three commands in a single block without any comments:\n```bash'}], 'stop_sequences': ['```']}


'aws s3 ls\naws ec2 describe-instances\naws lambda list-functions'

In [26]:
from IPython.display import Markdown
Markdown(text)


aws s3 ls

aws ec2 describe-instances

aws iam list-users
